In [1]:
import pandas as pd
import numpy as np
import sqlite3
import re

Objectives to fix:
1. city codes [i gave up on this, i tried using a geocode API, but its not possible because with over 100 addresses, it only identified 7 addresses. It is not worth it]
2. update the relationships, so separate out the contacts

# 1. Opening the File and Renaming Columns

In [36]:
eg_df = pd.read_excel("2024 EGAC.xlsx", sheet_name=0)
ac_df = pd.read_excel("2024 EGAC.xlsx", skiprows=1, sheet_name=1)

In [38]:
# Rename
OLD_NEW_NAMES_MAPPING = {
    'Provider Name': 'Institution Name',
    'Assessment Center': 'Institution Name',
    'Class Name': 'Class',
    'Region Name': 'Region',
    'Province Name': 'Province',
    'City Name': 'City',
    'focal': 'Center Manager',
    'phone': 'Phone',
    'e mail': 'Email',
    'Numberof Accredited Ass Qual': 'Number of Accredited Ass Qual'
}

eg_df = eg_df.rename(columns=OLD_NEW_NAMES_MAPPING).copy()


ac_df = ac_df.rename(columns=OLD_NEW_NAMES_MAPPING).copy()

# Add if a center is for development or assessment
eg_df['Is_Development'] = 1
eg_df['Is_Assessment'] = 0
ac_df['Is_Development'] = 0
ac_df['Is_Assessment'] = 1

# merge on the new names
egac_df = pd.concat([eg_df, ac_df])

# Clean up the City
egac_df['City'] = egac_df['City'].str.strip()

# Fix the typo of leve and standardize the casing and get the nc type
egac_df['Qualification'] = egac_df['Qualification'].str.replace(r'\bleve\b', 'Level', flags=re.IGNORECASE, regex=True)
egac_df['NC Type'] = egac_df['Qualification'].str.extract(r'\b((NC|Level).*)$')[0].str.strip()

# convert the statistics to int
statistics = ['Trainer Count','Enrolled', 'Graduates', 'Number of Accredited Ass Qual', 'Assessed','Certified',]
egac_df[statistics] = egac_df[statistics].replace(np.nan, 0).astype(int)

The goal of this mapping is to **standardize inconsistent institutional labels** into a smaller, analytically meaningful set of categories. Several transformations were made based on data patterns and domain assumptions:

* *Normalization of plural and variant forms*
  Labels such as `"TVIs"` → `"TVI"` and `"HEIs"` → `"HEI"` were simplified to ensure consistency across records.

* *Merging synonymous or equivalent labels*
  For example, `"Local Government Unit (LGU)"`, `"LGU"`, and `"LCU"` were all mapped to `"LGU-operated"` since they represent institutions managed by local government entities.

* *Collapsing overly specific subtypes into broader categories*
  Labels like `"TTI (PTC)"`, `"TTI (RTC)"`, and `"TTI (School)"` were grouped under `"TTI"` to reduce unnecessary granularity that does not materially affect analysis.

* *Handling rare or one-off labels*
  `"Specialized Training Center"` was mapped to `"TTI"` because it appears to refer to a **single institution** (e.g., Language Skills Institute in Zamboanga). Since it functionally behaves like a TESDA training institution, it is more consistent to classify it under `"TTI"` rather than maintain a separate category.

* *Assumption-based classification*
  `"Cooperative"` and `"GOCC/GFI"` were mapped to `"TVI"` based on the assumption that they operate as **training providers**, though this may require validation.
  For the six institutions that were cooperatives, The only institution for GOCC/GFI is BISLIG CITY WATER DISTRICT TRAINING CENTER, and that is private. 

* *Temporary groupings requiring refinement*
  Categories like `"Farm School"`, `"DepEd Supv."`, and `"National Government Agency (NGA)"` were assigned to `"Others"` or `"Other"` as placeholders. These should be revisited if finer classification becomes necessary.



In [39]:
# Standardized mapping of institution class labels
OLD_TO_NEW_CLASS_MAPPING = {
    "TVIs": "TVI",
    "TVI": "TVI",
    "HEIs": "HEI",
    "Higer Education Institution (HEI)": "HEI",

    "SUCs": "SUC",
    "State University and Colleges (SUC)": "SUC",

    "LGU": "LGU-operated",
    "Local Government Unit (LGU)": "LGU-operated",
    "LCU": "LGU-operated",

    "Enterprise/Company": "Company-based",
    "Cooperative": "TVI",  # Assumed private training provider

    "TTI (PTC)": "TTI",
    "TTI (RTC)": "TTI",
    "TTI (School)": "TTI",
    "TESDA Technology Institution (TTI)": "TTI",
    "Specialized Training Center": "TTI",

    "NGO/Foundation": "NGO/Foundation",
    "Non-Government Organization (NGO)": "NGO/Foundation",

    "GOCC/GFI": "TVI",  # Based on observed institution classification

    "Farm School": "Others",  # To be refined
    "Others": "Others",
    "Other": "Others",

    "DepEd Supv.": "Other",  # To be refined
    "National Government Agency (NGA)": "Other",  # To be refined
}

egac_df['Class'] = egac_df['Class'].map(OLD_TO_NEW_CLASS_MAPPING)
egac_df.fillna('Others', inplace=True)

# 2. Creating the Schema

In [66]:
# the easier ones (no foreign keys)
institution_creation = """
CREATE TABLE IF NOT EXISTS Institution (
    institution_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    class TEXT
);
"""

location_creation = """
CREATE TABLE IF NOT EXISTS Location (
    location_id INTEGER PRIMARY KEY,
    city TEXT NOT NULL,
    province TEXT NOT NULL,
    region TEXT NOT NULL
);
"""

qualification_creation = """
CREATE TABLE IF NOT EXISTS Qualification (
    qualification_name TEXT PRIMARY KEY,
    nc_type TEXT NOT NULL
);
"""

contact_creation = """
CREATE TABLE IF NOT EXISTS Contact (
    contact_id INTEGER PRIMARY KEY,
    focal_person TEXT NOT NULL,
    phone TEXT NOT NULL,
    email TEXT NOT NULL,
    address TEXT NOT NULL
);
"""

# the harder one (has foreign keys)
center_creation = """
CREATE TABLE IF NOT EXISTS Center (
    center_id INTEGER PRIMARY KEY,

    institution_id INTEGER NOT NULL,
    location_id INTEGER NOT NULL,

    FOREIGN KEY (institution_id) REFERENCES Institution(institution_id),
    FOREIGN KEY (location_id) REFERENCES Location(location_id)
);
"""

program_creation = """
CREATE TABLE IF NOT EXISTS Program (
    program_id INTEGER PRIMARY KEY,

    center_id INTEGER NOT NULL,
    qualification_name TEXT NOT NULL,

    is_development INTEGER NOT NULL DEFAULT 0,
    is_assessment INTEGER NOT NULL DEFAULT 0,

    num_trainer INTEGER,
    num_enrolled INTEGER,
    num_graduated INTEGER,
    num_assessor INTEGER,
    num_assessed INTEGER,
    num_certified INTEGER,

    FOREIGN KEY (center_id) REFERENCES Center(center_id),
    FOREIGN KEY (qualification_name) REFERENCES Qualification(qualification_name)
);
"""

program_contact_creation = """
CREATE TABLE IF NOT EXISTS ProgramContact (
    program_id INTEGER NOT NULL,
    contact_id INTEGER NOT NULL,

    is_development INTEGER,
    is_assessment INTEGER,

    PRIMARY KEY (program_id, contact_id),

    FOREIGN KEY (program_id) REFERENCES Program(program_id),
    FOREIGN KEY (contact_id) REFERENCES Contact(contact_id)
);
"""

In [67]:
schema = [
    institution_creation,
    location_creation,
    qualification_creation,
    contact_creation,
    center_creation,
    program_creation,
    program_contact_creation
]

conn = sqlite3.connect('egac2.db')
cursor = conn.cursor()

for table in schema:
    cursor.execute(table)

# 3a. Populating the Data for Easier Data

In [90]:
# Institution
insti_cols = {'index':'institution_id', 'Institution Name': 'name', 'Class': 'class'}
institution_df = (
    egac_df[['Institution Name', 'Class']]
    .drop_duplicates()
    .reset_index()
    .rename(columns=insti_cols)
)
institution_df.to_sql(
    'Institution',
    con=conn,
    if_exists='replace',
    index=False
)

# Location
loc_cols = {'index':'location_id', 'City': 'city', 'Province': 'province', 'Region':'region'}
location_df = (
    egac_df[['City', 'Province', 'Region']]
    .replace(
        'Autonomous Region in Muslim Mindanao (ARMM)',
        'Bangsamoro Autonomous Region In Muslim Mindanao (BARMM)')
    .drop_duplicates()
    .reset_index()
    .rename(columns=loc_cols)
)
location_df.to_sql(
    'Location',
    con=conn,
    if_exists='replace',
    index=False
)

# Qualification
qualification_cols = {'Qualification': 'qualification', 'NC Type': 'nc_type'}
qualification_df = (
    egac_df[qualification_cols.keys()]
    .drop_duplicates()
    .rename(columns=qualification_cols)
)
qualification_df.to_sql(
    'Qualification',
    con=conn,
    if_exists='replace',
    index=False
)

# Contacts
contact_cols = {
    'index': 'contact_id',
    'Center Manager': 'focal_person',
    'Phone': 'phone',
    'Email': 'email',
    'Address': 'address'
}

contact_df = egac_df[['Center Manager', 'Phone', 'Email', 'Address']].copy()

norm_cols = {
    'Center Manager': 'focal_norm',
    'Phone': 'phone_norm',
    'Email': 'email_norm',
    'Address': 'address_norm'
}

for col, norm_col in norm_cols.items():
    contact_df[norm_col] = (
        contact_df[col]
        .str.strip()
        .str.lower()
    )

contact_df = contact_df.drop_duplicates(
    subset=norm_cols.values()
).reset_index()

contact_df = contact_df.rename(columns=contact_cols)

sql_contact_df = contact_df[[
    'contact_id',
    'focal_person',
    'phone',
    'email',
    'address'
]]

sql_contact_df.to_sql(
    'Contact',
    con=conn,
    if_exists='replace',
    index=False
)

5791

In [91]:
# Build the lookup tables
inst_lookup = institution_df[['institution_id', 'name']]
loc_lookup = location_df[['location_id', 'city', 'province', 'region']]
contact_lookup = contact_df.copy()

# 3b. Populating the Database (with Foreign Keys)

In [92]:
center_df = egac_df[['Institution Name', 'City', 'Province', 'Region']].drop_duplicates()

# map institution_id
center_df = center_df.merge(
    inst_lookup,
    left_on='Institution Name',
    right_on='name'
)

# map location_id
center_df = center_df.merge(
    loc_lookup,
    left_on=['City', 'Province', 'Region'],
    right_on=['city', 'province', 'region']
)

# assign center_id
center_df = center_df.reset_index().rename(columns={'index': 'center_id'})
sql_center_df = center_df[['center_id', 'institution_id', 'location_id']].copy()

sql_center_df.to_sql(
    'Center',
    con=conn,
    if_exists='replace',
    index=False
)

5434

In [93]:
program_df = egac_df.copy()

# merge program_df to get center id
program_df = egac_df.merge(
    center_df,
    how="left",
    on=['Institution Name', 'City', 'Province', 'Region']
)

sql_program_df = program_df.groupby(
    ['center_id', 'Qualification'],
    as_index=False
).agg({
    'Is_Development': 'max',
    'Is_Assessment': 'max',
    'Trainer Count': 'max',
    'Enrolled': 'sum',
    'Graduates': 'sum',
    'Number of Accredited Ass Qual': 'max',
    'Assessed': 'sum',
    'Certified': 'sum'
})
sql_program_df = sql_program_df.reset_index().rename(columns={'index':'program_id'}).copy()
sql_program_df.center_id = sql_program_df.center_id.astype(int)
OLD_NEW_COLS = {
    'Qualification': 'qualification_name',
    'Is_Development': 'is_development',
    'Is_Assessment': 'is_assessment',
    'Trainer Count': 'num_trainer',
    'Enrolled': 'num_enrolled',
    'Graduates': 'num_graduated',
    'Number of Accredited Ass Qual': 'num_assessor',
    'Assessed':'num_assessed',
    'Certified': 'num_certified'
 }
sql_program_df.rename(columns=OLD_NEW_COLS, inplace=True)
sql_program_df.to_sql(
    'Program',
    con=conn,
    if_exists='replace',
    index=False)

18558

In [95]:
contact_df

,contact_id,focal_person,phone,email,address,focal_norm,phone_norm,email_norm,address_norm
0,0,Marcelino C. Anino,09286802859,iti_maragusan@yahoo.com,"Poblacion, Maragusan, Compostela Valley Province",marcelino c. anino,09286802859,iti_maragusan@yahoo.com,"poblacion, maragusan, compostela valley province"
1,1,ELDON M. PARREÑO,09126003270/09631260723,itcdrggdo@gmail.com,"Governor Generoso, Davao Oriental",eldon m. parreño,09126003270/09631260723,itcdrggdo@gmail.com,"governor generoso, davao oriental"
2,2,"Cora Merian C. Toralba, CPA",074-442-3313,informaticsbaguioregistrar@gmail.com,"65 Bonifacio St., 3rd Flr. Decibar Bldg., Boni...","cora merian c. toralba, cpa",074-442-3313,informaticsbaguioregistrar@gmail.com,"65 bonifacio st., 3rd flr. decibar bldg., boni..."
3,3,ELIAS P. MAHUMAS JR,09286802859,mit.informatic@gmail.com,"Binuangan, Maco, Compostela Valley Province",elias p. mahumas jr,09286802859,mit.informatic@gmail.com,"binuangan, maco, compostela valley province"
4,4,"Ma. Cristina B. Orbecido, Ph.D",34-4356092,nolitc.tesda@gmail.com,"Paglaum Sports Complex, Hernaez St., Bacolod City","ma. cristina b. orbecido, ph.d",34-4356092,nolitc.tesda@gmail.com,"paglaum sports complex, hernaez st., bacolod city"
...,...,...,...,...,...,...,...,...,...
5786,33508,Rosanna G. Siray,+60392023756,malaysia@owwa.gov.ph,"95 Jalan Perkasa, Taman Maluri Cheras, 55100 K...",rosanna g. siray,+60392023756,malaysia@owwa.gov.ph,"95 jalan perkasa, taman maluri cheras, 55100 k..."
5787,33511,Nerissa L. Baldemor,00966502850944,mwo_riyadh@dmw.gov.ph,"D3 Collector Road C. Diplomatic Quarter, P.o. ...",nerissa l. baldemor,00966502850944,mwo_riyadh@dmw.gov.ph,"d3 collector road c. diplomatic quarter, p.o. ..."
5788,33518,Ms. Shirley Abaja,Others,shirleyabaja@aimsintl.com.sg,33 Ubi Avenue 3 # 06-68 Tower A Vertex Bldg. S...,ms. shirley abaja,others,shirleyabaja@aimsintl.com.sg,33 ubi avenue 3 # 06-68 tower a vertex bldg. s...
5789,33521,Satyaprakash Tiwari,+6591504488,tiwari91504488@gmail.com,"3 Chin Cheng Avenue , Singapore",satyaprakash tiwari,+6591504488,tiwari91504488@gmail.com,"3 chin cheng avenue , singapore"


In [96]:
# merge to get contact id
program_contact_df = program_df.merge(
    contact_df,
    how='left',
    left_on=['Center Manager', 'Phone', 'Email', 'Address'],
    right_on=['focal_person', 'phone', 'email', 'address']
)

# Build lookup for program_id
program_lookup = sql_program_df[['program_id', 'center_id', 'qualification_name']].copy()

program_contact_df = program_contact_df.merge(
    program_lookup,
    how='left',
    left_on=['center_id', 'Qualification'],
    right_on=['center_id', 'qualification_name']
)

program_contact_df = program_contact_df[
    ['program_id', 'contact_id', 'Is_Development', 'Is_Assessment']
]

program_contact_df = (
    program_contact_df
    .dropna(subset=['program_id', 'contact_id'])
    .drop_duplicates()
    .groupby(['program_id', 'contact_id'], as_index=False)
    .agg({
        'Is_Development': 'max',
        'Is_Assessment': 'max'
    })
)
program_contact_df

,program_id,contact_id,Is_Development,Is_Assessment
0,0.0,0.0,1,0
1,1.0,0.0,1,0
2,2.0,0.0,1,0
3,3.0,1.0,1,0
4,4.0,1.0,1,0
...,...,...,...,...
21762,18553.0,33521.0,0,1
21763,18554.0,33526.0,0,1
21764,18555.0,33526.0,0,1
21765,18556.0,33526.0,0,1


In [ ]:
conn = sqlite3.connect("egac.db")

query = """
SELECT 
    i.id AS institution_id,
    i.name AS institution_name,
    i.class_name AS class,
    COUNT(DISTINCT c.id) AS num_centers,
    COUNT(DISTINCT p.qualification_name) AS num_qualifications,
    COALESCE(SUM(p.num_trainer), 0) AS total_trainers,
    COALESCE(SUM(p.num_enrolled), 0) AS total_enrolled,
    COALESCE(SUM(p.num_graduated), 0) AS total_graduates,
    COALESCE(SUM(p.num_accredited_ass), 0) AS total_accredited_assessors,
    COALESCE(SUM(p.num_assessed), 0) AS total_assessed,
    COALESCE(SUM(p.num_certified), 0) AS total_certified
FROM institution i
LEFT JOIN center c ON i.id = c.institution_id
LEFT JOIN program p ON c.id = p.center_id
GROUP BY i.id, i.name, i.class_name
ORDER BY i.name;
"""

df_stats = pd.read_sql_query(query, conn)
conn.close()


In [ ]:
df_stats[df_stats['institution_name'] == 'PHILTECH INSTITUTE OF ARTS AND TECHNOLOGY INC.']

,institution_id,institution_name,class,num_centers,num_qualifications,total_trainers,total_enrolled,total_graduates,total_accredited_assessors,total_assessed,total_certified
3073,3752,PHILTECH INSTITUTE OF ARTS AND TECHNOLOGY INC.,None,2,7,0,0,0,7,273,208
